In [1]:
from pathlib import Path
from datetime import datetime
import sys
import os
current_dir = os.getcwd()
project_root = os.path.join(current_dir, '..', '..')  # Adjust based on your notebook location
sys.path.append(project_root)
from src.models.xgboost_randomforrest_model_v2 import NFLModelV2
import pandas as pd
from src.utils.db_utils import get_connection, execute_query
import numpy as np
engine = get_connection()


Successfully connected to the database!


In [2]:
TARGETS = [
    "spread",
    "total_points",
    "binary_spread_label",
    "binary_ou_label"
]

In [3]:
target=TARGETS[0]
model = NFLModelV2(target=target)
model.load_games(start_season=2020)
# Build offensive + defensive + differential features
model.build_feature_matrices(include_defense=True, include_differentials=True)
model.build_dataset()

Successfully connected to the database!


,gamesummaryid,season,week,hometeamid,awayteamid,homescore,awayscore,spread,total_points,binary_spread_label,...,opp_third_down_efficiency__diff_roll2,opp_third_down_efficiency__off_roll5,opp_third_down_efficiency__def_roll5,opp_third_down_efficiency__diff_roll5,opp_third_down_efficiency__off_roll10,opp_third_down_efficiency__def_roll10,opp_third_down_efficiency__diff_roll10,opp_third_down_efficiency__sos_ratio,opp_third_down_efficiency__sos_inv_ratio,opp_third_down_efficiency__off_hist_z
0,gs-202001ARISFO,2020,1,SFO,ARI,20,24,-4,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gs-202001CHIDET,2020,1,DET,CHI,23,27,-4,50,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gs-202001CLEBAL,2020,1,BAL,CLE,38,6,32,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gs-202001DALLAR,2020,1,LAR,DAL,20,17,3,37,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gs-202001GNBMIN,2020,1,MIN,GNB,34,43,-9,77,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1434,gs-202502PHIKAN,2025,2,KAN,PHI,17,20,-3,37,1,...,0.038462,0.389441,0.489625,-0.100185,0.398210,0.453146,-0.054936,1.071854,0.932963,0.927601
1435,gs-202502SEAPIT,2025,2,PIT,SEA,17,31,-14,48,0,...,-0.160606,0.347552,0.450256,-0.102704,0.374033,0.395700,-0.021667,0.962991,1.038432,-0.859500
1436,gs-202502SFONOR,2025,2,NOR,SFO,21,26,-5,47,1,...,-0.100962,0.428846,0.526609,-0.097763,0.433514,0.432856,0.000658,1.145320,0.873119,0.993880
1437,gs-202502TAMHOU,2025,2,HOU,TAM,19,20,-1,39,1,...,0.076729,0.578968,0.390521,0.188448,0.498077,0.375114,0.122963,1.082527,0.923765,1.155311


In [4]:
model.build_feature_matrices(include_defense=True, include_differentials=True)
model.build_dataset()

,gamesummaryid,season,week,hometeamid,awayteamid,homescore,awayscore,spread,total_points,binary_spread_label,...,opp_third_down_efficiency__diff_roll2,opp_third_down_efficiency__off_roll5,opp_third_down_efficiency__def_roll5,opp_third_down_efficiency__diff_roll5,opp_third_down_efficiency__off_roll10,opp_third_down_efficiency__def_roll10,opp_third_down_efficiency__diff_roll10,opp_third_down_efficiency__sos_ratio,opp_third_down_efficiency__sos_inv_ratio,opp_third_down_efficiency__off_hist_z
0,gs-202001ARISFO,2020,1,SFO,ARI,20,24,-4,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gs-202001CHIDET,2020,1,DET,CHI,23,27,-4,50,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gs-202001CLEBAL,2020,1,BAL,CLE,38,6,32,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gs-202001DALLAR,2020,1,LAR,DAL,20,17,3,37,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gs-202001GNBMIN,2020,1,MIN,GNB,34,43,-9,77,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1434,gs-202502PHIKAN,2025,2,KAN,PHI,17,20,-3,37,1,...,0.038462,0.389441,0.489625,-0.100185,0.398210,0.453146,-0.054936,1.071854,0.932963,0.927601
1435,gs-202502SEAPIT,2025,2,PIT,SEA,17,31,-14,48,0,...,-0.160606,0.347552,0.450256,-0.102704,0.374033,0.395700,-0.021667,0.962991,1.038432,-0.859500
1436,gs-202502SFONOR,2025,2,NOR,SFO,21,26,-5,47,1,...,-0.100962,0.428846,0.526609,-0.097763,0.433514,0.432856,0.000658,1.145320,0.873119,0.993880
1437,gs-202502TAMHOU,2025,2,HOU,TAM,19,20,-1,39,1,...,0.076729,0.578968,0.390521,0.188448,0.498077,0.375114,0.122963,1.082527,0.923765,1.155311


In [5]:
def gamesummary():

    query = """
    SELECT *
    FROM stats.teamstats
    WHERE season > '2023' and teamid='ARI'

    """
    return execute_query(query)
tables_df = gamesummary()

Successfully connected to the database!


In [6]:
def add_game_week_qid(df, season_col='season', week_col='week', qid_col='qid'):
    """
    Add a query ID (qid) for each unique combination of season and week.
    All games in the same season and week will have the same qid.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing NFL game data
    season_col : str, default 'season'
        Name of the column containing season information
    week_col : str, default 'week'
        Name of the column containing week information
    qid_col : str, default 'qid'
        Name of the new column to create for query IDs
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with added qid column
        
    Examples:
    ---------
    >>> df = pd.DataFrame({
    ...     'season': [2023, 2023, 2023, 2024, 2024, 2024],
    ...     'week': [1, 1, 2, 1, 1, 2],
    ...     'team': ['A', 'B', 'C', 'D', 'E', 'F']
    ... })
    >>> df_with_qid = add_game_week_qid(df)
    >>> print(df_with_qid)
       season  week team  qid
    0    2023     1    A    0
    1    2023     1    B    0
    2    2023     2    C    1
    3    2024     1    D    2
    4    2024     1    E    2
    5    2024     2    F    3
    """
    
    # Create a copy to avoid modifying the original DataFrame
    df_copy = df.copy()
    
    # Check if required columns exist
    if season_col not in df_copy.columns:
        raise ValueError(f"Column '{season_col}' not found in DataFrame")
    if week_col not in df_copy.columns:
        raise ValueError(f"Column '{week_col}' not found in DataFrame")
    
    # Create a unique identifier for each season-week combination
    # Sort by season and week to ensure chronological order
    unique_combinations = (df_copy[[season_col, week_col]]
                          .drop_duplicates()
                          .sort_values([season_col, week_col])
                          .reset_index(drop=True))
    
    # Assign sequential qid values
    unique_combinations[qid_col] = range(len(unique_combinations))
    
    # Merge back to original DataFrame
    df_with_qid = df_copy.merge(
        unique_combinations, 
        on=[season_col, week_col], 
        how='left'
    )
    
    # Ensure qid is integer type
    df_with_qid[qid_col] = df_with_qid[qid_col].astype(int)
    
    print(f"Added {qid_col} column with {df_with_qid[qid_col].nunique()} unique season-week combinations")
    print(f"QID range: {df_with_qid[qid_col].min()} to {df_with_qid[qid_col].max()}")
    
    return df_with_qid

data_w_qid=add_game_week_qid(model._dataset)
data_w_qid


Added qid column with 111 unique season-week combinations
QID range: 0 to 110


,gamesummaryid,season,week,hometeamid,awayteamid,homescore,awayscore,spread,total_points,binary_spread_label,...,opp_third_down_efficiency__off_roll5,opp_third_down_efficiency__def_roll5,opp_third_down_efficiency__diff_roll5,opp_third_down_efficiency__off_roll10,opp_third_down_efficiency__def_roll10,opp_third_down_efficiency__diff_roll10,opp_third_down_efficiency__sos_ratio,opp_third_down_efficiency__sos_inv_ratio,opp_third_down_efficiency__off_hist_z,qid
0,gs-202001ARISFO,2020,1,SFO,ARI,20,24,-4,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,gs-202001CHIDET,2020,1,DET,CHI,23,27,-4,50,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,gs-202001CLEBAL,2020,1,BAL,CLE,38,6,32,44,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,gs-202001DALLAR,2020,1,LAR,DAL,20,17,3,37,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,gs-202001GNBMIN,2020,1,MIN,GNB,34,43,-9,77,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1434,gs-202502PHIKAN,2025,2,KAN,PHI,17,20,-3,37,1,...,0.389441,0.489625,-0.100185,0.398210,0.453146,-0.054936,1.071854,0.932963,0.927601,110
1435,gs-202502SEAPIT,2025,2,PIT,SEA,17,31,-14,48,0,...,0.347552,0.450256,-0.102704,0.374033,0.395700,-0.021667,0.962991,1.038432,-0.859500,110
1436,gs-202502SFONOR,2025,2,NOR,SFO,21,26,-5,47,1,...,0.428846,0.526609,-0.097763,0.433514,0.432856,0.000658,1.145320,0.873119,0.993880,110
1437,gs-202502TAMHOU,2025,2,HOU,TAM,19,20,-1,39,1,...,0.578968,0.390521,0.188448,0.498077,0.375114,0.122963,1.082527,0.923765,1.155311,110


In [17]:
model.feature_columns

['completion_efficiency__def_hist',
 'completion_efficiency__def_roll10',
 'completion_efficiency__def_roll2',
 'completion_efficiency__def_roll5',
 'completion_efficiency__diff_hist',
 'completion_efficiency__diff_roll10',
 'completion_efficiency__diff_roll2',
 'completion_efficiency__diff_roll5',
 'completion_efficiency__off_hist',
 'completion_efficiency__off_hist_z',
 'completion_efficiency__off_roll10',
 'completion_efficiency__off_roll2',
 'completion_efficiency__off_roll5',
 'completion_efficiency__sos_inv_ratio',
 'completion_efficiency__sos_ratio',
 'first_downs__def_hist',
 'first_downs__def_roll10',
 'first_downs__def_roll2',
 'first_downs__def_roll5',
 'first_downs__diff_hist',
 'first_downs__diff_roll10',
 'first_downs__diff_roll2',
 'first_downs__diff_roll5',
 'first_downs__off_hist',
 'first_downs__off_hist_z',
 'first_downs__off_roll10',
 'first_downs__off_roll2',
 'first_downs__off_roll5',
 'first_downs__sos_inv_ratio',
 'first_downs__sos_ratio',
 'fourth_down_attempts

In [15]:
from sklearn.model_selection import TimeSeriesSplit
tscv=TimeSeriesSplit(n_splits=4)
season_lookback=2
for train_idx, test_idx in tscv.split(data_w_qid["qid"]):
    train_idx_unique=np.unique(data_w_qid.iloc[train_idx]["qid"])
    train_idx_int=int(train_idx_unique.shape[0]*.2)
    train_qids=train_idx_unique[:-train_idx_int]
    val_qids=train_idx_unique[-train_idx_int:]
    training_data=data_w_qid[data_w_qid["qid"].isin(train_qids)]
    
    val_data=data_w_qid[data_w_qid["qid"].isin(val_qids)]
    current_season=np.max(val_data["season"])
    lookback_season=current_season-season_lookback
    
    training_data=training_data.iloc[np.where(training_data["season"]>=lookback_season)]
    



In [16]:
training_data

,gamesummaryid,season,week,hometeamid,awayteamid,homescore,awayscore,spread,total_points,binary_spread_label,...,opp_third_down_efficiency__off_roll5,opp_third_down_efficiency__def_roll5,opp_third_down_efficiency__diff_roll5,opp_third_down_efficiency__off_roll10,opp_third_down_efficiency__def_roll10,opp_third_down_efficiency__diff_roll10,opp_third_down_efficiency__sos_ratio,opp_third_down_efficiency__sos_inv_ratio,opp_third_down_efficiency__off_hist_z,qid
553,gs-202201BALNYJ,2022,1,NYJ,BAL,9,24,-15,33,1,...,0.333450,0.464530,-0.131080,0.361427,0.485562,-0.124134,0.964830,1.036452,0.593399,43
554,gs-202201BUFLAR,2022,1,LAR,BUF,10,31,-21,41,1,...,0.562698,0.280952,0.281746,0.481899,0.307987,0.173912,1.306038,0.765675,1.842939,43
555,gs-202201CLECAR,2022,1,CAR,CLE,24,26,-2,50,0,...,0.453512,0.435365,0.018148,0.368411,0.410320,-0.041909,0.992961,1.007089,0.400678,43
556,gs-202201DENSEA,2022,1,SEA,DEN,17,16,1,33,1,...,0.412045,0.446154,-0.034108,0.423592,0.427253,-0.003661,0.904919,1.105072,-0.676498,43
557,gs-202201GNBMIN,2022,1,MIN,GNB,23,7,16,30,1,...,0.476422,0.369231,0.107192,0.445356,0.389767,0.055589,1.261428,0.792753,1.846207,43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
938,gs-202307LVRCHI,2023,7,CHI,LVR,30,12,18,42,0,...,0.342358,0.489384,-0.147026,0.392348,0.499616,-0.107269,0.926947,1.078810,-0.161534,71
939,gs-202307MIAPHI,2023,7,PHI,MIA,31,17,14,48,1,...,0.433803,0.436196,-0.002392,0.374258,0.393464,-0.019206,0.976029,1.024560,-0.583156,71
940,gs-202307PITLAR,2023,7,LAR,PIT,17,24,-7,41,0,...,0.369524,0.382712,-0.013189,0.467976,0.358652,0.109324,1.094441,0.913708,0.190066,71
941,gs-202307SFOMIN,2023,7,MIN,SFO,22,17,5,39,1,...,0.436035,0.453766,-0.017731,0.439871,0.423852,0.016019,1.079379,0.926458,0.349679,71


In [11]:
tables_df.to_excel("ARI data.xlsx")

ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
model._dataset

In [ ]:
model_data=model._dataset.set_index("gamesummaryid")
tables_df=tables_df.set_index("gamesummaryid")
# ari_idx=np.where(==tables_df["gamesummaryid"].values)[0]
model_data.loc[tables_df.index].to_excel("model data.xlsx")

In [ ]:
model_data.loc[tables_df.index,"total_yards__off_hist"]

In [ ]:
tables_df["total_yards"].expanding().mean().shift(1)==model_data.loc[tables_df.index,"total_yards__off_hist"]